# Multi-candidate attribution — the interface (D207)

**What this is.** One Run All produces the single attribution number `N` for the
30–60° DB-turn band, with its composition printed underneath and the identified
set beside the frozen estimate. It is the spec's §0 deliverable
(`notes/multi-candidate-spec.md`), assembled from four slots:

```
N = N_frozen + N_geo(5) + N_geo(8) + N_routeA
```

**The order is fixed and executed.** `frozen → geo(5) → geo(8) → Route A`,
each pass only on the turn keys the earlier ones left — computed by
`aa.attribution_pipeline`, in one place, for the notebook and the script alike.
`precedence.csv` prints the four slots with their own denominators, the number of
turns in more than one slot (must be 0), and the overlap between Route A and the
radius slots: the sum of both passes is an upper bound on `N`, never `N`.

**What it is not.** The dataset records no opponent identifier. `N_frozen` is the
frozen elimination match and is the only slot labelled `RULE`; every other slot is
an **INFERENCE** and no output here says which man was covered. Nothing here enters
`TASK.md`, the band table, the six figures or the report draft.

**How to run it.** Edit the one config cell below, then Run All.
The first printed block says `TRACKING: FOUND`, with every file, its size,
the rows read and the seasons, or `TRACKING: NONE FOUND`, naming each
directory searched and what was in it. With no tracking the geometric slots
report `NOT EVALUABLE` and contribute zero; put the full game-tracking files
in `data/raw` (any `*tracking*.csv` name is found), or point `TRACKING` at
them, and they are computed.

**What it writes.** Run All writes all fifteen tables it computes into
`outputs/ambiguity_attribution` — the same directory the script writes — plus
`manifest.json` with a UTC timestamp, the sha256 of every input and output and
the tracking status. The path and the file count are printed at the top and
again beside `N`, so the files on disk can be matched to the run that made
them.

**If a run stops.** `run_status.txt` in that same folder is created and
written before anything is measured, then appended to at each step. A run
that dies part-way leaves the last step it reached there, so a failed run
is diagnosable from disk and not from console output. There is no CLI and
nothing is installed.

In [ ]:
# ---- CONFIG — the only cell you edit -----------------------------------
# None means "the repo default". A path is used as given.
TURNS       = None   # -> data/interim/full_run_20261111/bdb27_full_db_turns.parquet
PLAYER_PLAY = None   # -> data/raw/player_play.csv
PLAYERS     = None   # -> data/raw/players.csv

# []  -> auto-discovery: every *.csv under data/raw, searched recursively, whose
#        NAME contains "tracking" in any case and not "sample" or "combine".
#        Every directory config.yaml names under paths is searched too, so a
#        per-season or differently-named export is found without editing this
#        cell. The block printed below names each directory searched and every
#        file found, so a naming mismatch is visible in one glance. "sample" is
#        excluded because a truncation cannot answer the distance questions and
#        must not stand in for the full file.
# [...] -> full tracking file(s) or folder(s), used as given: a folder is
#        globbed the same way, a file is taken as given whatever its name (that
#        is how the sample file can be pointed at deliberately, and how it then
#        prints its truncation).
TRACKING    = []

RADII           = (5.0, 8.0)   # the D204 sensitivity radii, unchanged
SEED            = 20261117     # AC3's fixed seed
REPS            = None         # None -> aa.RANDOM_REPS (AC3's replicate count)

# Run All writes every table it computes. OUT_DIR is the SAME directory the
# script writes, so a file opened after Run All is this run's output and not a
# committed file from another machine. False keeps the tables in memory only.
WRITE_ARTEFACTS = True
OUT_DIR         = "outputs/ambiguity_attribution"

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def repo_root(start: Path) -> Path:
    """The nearest directory above ``start`` that carries ``src/bdb27``."""
    tried = []
    for p in [start, *start.parents]:
        tried.append(str(p))
        if (p / "src" / "bdb27" / "ambiguity_attribution.py").exists():
            return p
    raise FileNotFoundError(
        "cannot find the BDB27 repo root; looked in:\n  " + "\n  ".join(tried))


ROOT = repo_root(Path.cwd().resolve())
SRC = ROOT / "src"

# A pull-then-Run-All in a kernel that already imported bdb27 would otherwise
# hand back the module object built before the pull: Python caches imports, and
# a sys.path insert cannot invalidate that cache. notebook_boot.py is loaded
# from the file on disk (it imports nothing from bdb27); it then drops every
# cached bdb27 module and imports the files THIS checkout holds. It prints the
# file it loaded and the checkout it came from, so a wrong-folder run is
# visible on screen without any command line.
import importlib.util

BOOT_FILE = SRC / "bdb27" / "notebook_boot.py"
if not BOOT_FILE.is_file():
    raise RuntimeError(
        "the notebook cannot find " + str(BOOT_FILE) + " — it was opened "
        "outside a BDB27 checkout, or the copy is incomplete. Open "
        "notebooks/bdb27_multi_candidate.ipynb from the BDB27 folder and "
        "press Run All.")
_boot_spec = importlib.util.spec_from_file_location(
    "_bdb27_notebook_boot", BOOT_FILE)
boot = importlib.util.module_from_spec(_boot_spec)
_boot_spec.loader.exec_module(boot)

modules = boot.fresh_import(
    SRC, ["outcome_test", "geometric_match", "ambiguity_attribution"], root=ROOT)
aa = modules["ambiguity_attribution"]
ot = modules["outcome_test"]
gm = modules["geometric_match"]

# Every name this notebook calls on the package. If a pull left the checkout
# older than the notebook, this stops with a sentence naming the file it loaded
# and the checkout — never a bare AttributeError.
REQUIRED_SYMBOLS = {
    "ambiguity_attribution": [
        "RUN_STATUS_NAME", "write_run_status", "run_status_header",
        "note_run_status", "tracking_status", "tracking_banner",
        "ARTEFACT_TABLES", "RANDOM_REPS", "SPECS", "PAIRING_RULES",
        "connect", "funnel", "build_frame", "analysis_frame", "identified_set",
        "all_candidates_frame", "attribution_pipeline",
        "single_candidate_geometry", "candidate_distance_table",
        "within_radius_distribution", "margin_distribution", "variant_fit",
        "nearest_variant", "multi_turn_table", "composition",
        "random_assignments", "alignment_counts", "agreement",
        "write_artefacts", "write_manifest",
    ],
    "outcome_test": ["TURN_KEY"],
    "geometric_match": ["candidate_frame", "recover_break_frames"],
}
boot.require_symbols(modules, REQUIRED_SYMBOLS, SRC, root=ROOT)

TURNS = Path(TURNS) if TURNS else ROOT / "data/interim/full_run_20261111/bdb27_full_db_turns.parquet"
PLAYER_PLAY = Path(PLAYER_PLAY) if PLAYER_PLAY else ROOT / "data/raw/player_play.csv"
PLAYERS = Path(PLAYERS) if PLAYERS else ROOT / "data/raw/players.csv"
OUT = Path(OUT_DIR) if Path(OUT_DIR).is_absolute() else ROOT / OUT_DIR
REPS = int(REPS or aa.RANDOM_REPS)

# The run's own evidence trail, before anything is measured. The output folder
# is created and run_status.txt is written HERE, so a run that dies in a later
# cell still leaves the last step it reached on disk. Overwritten every run.
OUT.mkdir(parents=True, exist_ok=True)
STATUS_FILE = OUT / aa.RUN_STATUS_NAME
STATUS_STARTED = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
INPUTS = {"turns": TURNS, "player_play": PLAYER_PLAY, "players": PLAYERS,
          "tracking": TRACKING}
aa.write_run_status(STATUS_FILE, aa.run_status_header(
    ROOT, OUT, INPUTS, None, started=STATUS_STARTED), keep_progress=False)
aa.note_run_status(STATUS_FILE, "run started")

print("repo       ", ROOT)
print("turns      ", TURNS)
print("player_play", PLAYER_PLAY)

# The one block that says whether geometry can be measured at all. It is a
# banner, never an exception: missing tracking is printed, never raised.
TRACKING_STATUS = aa.tracking_status(TRACKING or None)
print()
print(aa.tracking_banner(TRACKING_STATUS))

# The full status header, now that discovery has finished; the marker above
# already guaranteed the file exists even if discovery had stalled.
aa.write_run_status(STATUS_FILE, aa.run_status_header(
    ROOT, OUT, INPUTS, TRACKING_STATUS, started=STATUS_STARTED))
aa.note_run_status(STATUS_FILE, "tracking searched")
print(f"status: {STATUS_FILE}")

# Where this run puts its files, in full, before anything is measured: after
# Run All the directory listing below is this run's output, not a leftover.
if WRITE_ARTEFACTS:
    print(f"outputs: {OUT} — this run writes "
          f"{len(aa.ARTEFACT_TABLES) + 2} files here "
          f"({len(aa.ARTEFACT_TABLES)} tables + run_status.txt + "
          f"manifest.json)")
else:
    print(f"outputs: {OUT} — WRITE_ARTEFACTS is False: nothing is written, "
          f"the {len(aa.ARTEFACT_TABLES)} tables stay in memory")


In [ ]:
con = aa.connect()
try:
    funnel = aa.funnel(TURNS, PLAYER_PLAY, con=con)
    single = aa.build_frame(TURNS, PLAYER_PLAY, status="single", con=con)
    multiple = aa.build_frame(TURNS, PLAYER_PLAY, status="multiple", con=con)
finally:
    con.close()

frame = aa.analysis_frame(single, multiple)
band_turns = int(funnel.loc[funnel["step"] == "band_turns", "n"].iloc[0])

print(funnel[["step", "n", "denominator", "matches"]].to_string(index=False))
print()
print(f"frozen frame  {len(single):,} turns  ·  ambiguous "
      f"{multiple[list(ot.TURN_KEY)].drop_duplicates().shape[0]:,} turns "
      f"({len(multiple):,} candidate rows)")

aa.note_run_status(
    STATUS_FILE, f"funnel ok — band {band_turns:,} turns · "
                 f"{len(single):,} single + {len(multiple):,} ambiguous rows")


In [ ]:
# O2 — the identified set beside the frozen-only estimate (label set here only)
bounds = pd.DataFrame([aa.identified_set(frame, spec) for spec in aa.SPECS])
primary = aa.identified_set(frame, "primary")

print(bounds[["spec", "beta_frozen", "ci_lo", "ci_hi", "n_frozen", "frozen_clusters",
              "beta_min", "beta_max", "width", "ci_width", "label"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
print()
print(f"Label, set by O2 alone: {primary['branch']}")
print(f"  frozen-only beta {primary['beta_frozen']:+.3f}  "
      f"95% CI [{primary['ci_lo']:+.3f}, {primary['ci_hi']:+.3f}]  "
      f"n={primary['n_frozen']:,}  clusters={primary['frozen_clusters']:,}")
print(f"  identified set [{primary['beta_min']:+.3f}, {primary['beta_max']:+.3f}]  "
      f"width {primary['width']:.3f} vs CI width {primary['ci_width']:.3f}")
print("  the set carries no sampling error and is never combined with the CI")

n_ambiguous_turns = multiple[list(ot.TURN_KEY)].drop_duplicates().shape[0]
print()
print(f"populations, stated rather than implied: the primary specification is fitted on "
      f"the {primary['n_frozen']:,} single-candidate turns; the variants on "
      f"{len(single):,} + {n_ambiguous_turns:,} = "
      f"{len(single) + n_ambiguous_turns:,} turns")

aa.note_run_status(STATUS_FILE, f"bounds ok — {len(bounds)} specifications")


In [ ]:
# §0 — the four slots, in the order the specification fixes them:
#     frozen  ->  geo(5)  ->  geo(8)  ->  Route A
# Each pass sees only what the passes before it left, and the four sets are then
# checked for disjointness rather than assumed disjoint. One call, shared with
# scripts/ambiguity_attribution.py, so the two surfaces cannot disagree about the
# order or about which turns each route was allowed to see (Amendment 3).
frame_all = aa.all_candidates_frame(TURNS, PLAYER_PLAY)
frozen_keys = list(map(tuple, single[list(ot.TURN_KEY)].drop_duplicates()
                       .to_numpy().tolist()))

pipe = aa.attribution_pipeline(TURNS, PLAYER_PLAY, frame_all,
                               frozen_keys=frozen_keys,
                               tracking=(TRACKING or None),
                               players_path=PLAYERS,
                               radii=RADII, band_turns=band_turns)
geo = pipe["geo"]
route_turns, route_audit = pipe["route_turns"], pipe["route_audit"]
route_reference_turns = pipe["route_reference_turns"]
route_reference_audit = pipe["route_reference_audit"]
route_eligible = pipe["route_eligible"]
route_overlap = pipe["overlap"]
precedence = pipe["precedence"]
assembled = pipe["assembled"]

print(f"geometry status : {geo['status']}")
print(f"turns considered {geo['n_turns_considered']:,}  ·  "
      f"N_geo({RADII[0]:g}) {geo['n_geo_5']:,}  ·  "
      f"N_geo({RADII[1]:g}) {geo['n_geo_8']:,}  ·  "
      f"taken by a radius {len(pipe['geo_taken_keys']):,}")
print(f"paths tried     : {geo['paths_tried']}")
if len(geo["rows"]):
    print("per-turn status : "
          + " · ".join(f"{k} {v:,}" for k, v in
                       geo["rows"]["geometry_status"].value_counts().items()))

# What the tracking file is, as a finding — never a reason to stop (AC8).
# These counts are the ones the banner in cell 2 already printed: one read of
# the files, one set of numbers, no second pass over a multi-gigabyte export.
if TRACKING_STATUS["status"] == "FOUND" and TRACKING_STATUS.get("rows") is not None:
    print()
    print(f"tracking file(s) {len(TRACKING_STATUS['files'])}: "
          f"{TRACKING_STATUS['rows']:,} rows · {TRACKING_STATUS['plays']:,} plays · "
          f"{TRACKING_STATUS['players']:,} players · "
          f"{TRACKING_STATUS['rows_per_play']:.2f} rows per play")
    if TRACKING_STATUS["rows"] == 1_048_575:
        print("  FINDING: 1,048,575 rows is a truncation, not a season — the "
              "geometric slots are a code-path check here, not a result.")

aa.note_run_status(
    STATUS_FILE, f"geometry {geo['status'].split(' — ')[0]} / "
                 f"{len(geo['rows']):,} rows")

print()
print(f"Route A eligible : {route_eligible:,} of {pipe['frame_turns']:,} turns with a "
      f"visible opposing route-runner, after {len(frozen_keys):,} frozen and "
      f"{len(pipe['geo_taken_keys']):,} taken by a radius")
print(f"                   (a Route A pass on the frozen rule alone would see "
      f"{pipe['route_eligible_reference']:,}; that pass is the reference below, "
      f"not the slot)")
print(route_audit.to_string(index=False))
print()
print(f"Route A attributed {len(route_turns):,} of {route_eligible:,} "
      f"({(len(route_turns) / route_eligible if route_eligible else 0):.1%})")
if len(route_turns):
    print(route_turns["attribution_label"].value_counts().to_string())
    print("coverage label of those turns: "
          + " · ".join(f"{k} {v:,}" for k, v in
                       route_turns["coverage_family"].fillna("MISSING")
                       .value_counts().items()))

aa.note_run_status(STATUS_FILE, f"route_a {len(route_turns)}")


In [ ]:
# Amendment 3 — the slots, disjoint, and the overlap stated rather than assumed.
# The reference pass is Route A on the frozen-excluded pool: the pass the
# pre-amendment code reported as if it were the slot. It is NOT a slot, it is
# added to nothing, and it is kept because §4's cross-check needs the turns both
# routes fire on and because the overlap is what reconciles a summed-passes
# number with N.
print("PRECEDENCE — frozen -> geo(5) -> geo(8) -> Route A, each with its own "
      "denominator")
show = precedence.copy()
show["n"] = show["n"].map(lambda v: f"{int(v):,}")
show["denominator"] = show["denominator"].map(
    lambda v: "n/a" if pd.isna(v) else f"{int(v):,}")
show["share"] = show["share"].map(lambda v: "n/a" if pd.isna(v) else f"{v:.1%}")
print(show[["item", "n", "denominator", "share", "note"]].to_string(index=False))
print()
print(f"disjoint: {pipe['disjoint']} · turns in more than one slot "
      f"{pipe['n_double_counted']:,} of N = {assembled['N']:,}")
print(f"Route A overlap with the radii: {route_overlap:,} of "
      f"{len(route_reference_turns):,} reference-pass turns")
print(f"  N = {assembled['N']:,} = {assembled['slots']['n_frozen']:,} frozen + "
      f"{assembled['slots']['n_geo_5']:,} geo(5) + "
      f"{assembled['slots']['n_geo_8']:,} geo(8) + "
      f"{assembled['slots']['n_route_a']:,} Route A")
if route_overlap:
    print(f"  = {assembled['slots']['n_frozen']:,} + "
          f"{assembled['slots']['n_geo_5']:,} + "
          f"{assembled['slots']['n_geo_8']:,} + "
          f"({len(route_reference_turns):,} reference - {route_overlap:,} overlap)")
else:
    print("  no radius took a turn Route A would also have taken, so the overlap is "
          "0 and the sum of both passes is not an over-count on this run")
print(f"the sum of both passes would be {pipe['N_if_both_passes_summed']:,} — an "
      f"upper bound on N, never N")

aa.note_run_status(
    STATUS_FILE, f"precedence disjoint={pipe['disjoint']} / overlap {route_overlap}")


In [ ]:
# The frozen 3-yd step on the SINGLE-candidate set — a check, not a recovery route
singles = aa.single_candidate_geometry(TURNS, single, tracking=(TRACKING or None))

print(f"single-candidate turns : {singles['n_turns']:,}")
print(f"status                 : {singles['status']}")
print(f"tracked                : {singles['n_tracked']:,} of {singles['n_turns']:,}"
      f" ({singles['n_tracked'] / singles['n_turns']:.1%})")
print(f"within {singles['radius_yd']:g} yd at the break frame: "
      f"{singles['n_within_radius']:,} of {singles['n_tracked']:,} tracked turns")
print(f"  D204 reported {singles['d204_reported']:,} on the full file from the laptop; "
      f"this run counts its own number and does not assume that one")
if len(singles["rows"]):
    print("per-turn status: "
          + " · ".join(f"{k} {v:,}" for k, v in
                       singles["rows"]["geometry_status"].value_counts().items()))
print()
print("These turns are already inside N_frozen: this is the frozen set's own 3-yd "
      "test, it adds nothing to N, and it is not a recovery route.")

aa.note_run_status(
    STATUS_FILE, f"singles {singles['status'].split(' — ')[0]} / "
                 f"{len(singles['rows']):,} rows")


## Amendment — within 3 yd, V4 nearest, and the margin (Conor, 2026-11-17)

Three questions, on the **870** multi-candidate turns only:

1. **Within-3-yd distribution.** Zero / exactly one / exactly two / three or more
   candidates inside 3 yd at the break frame, with counts and shares. This says
   whether the frozen rule's no-tie-break clause was protecting us from genuine
   ambiguity or discarding turns it could have resolved.
2. **V4 — nearest, labelled `TIE-BREAK`.** Where two or more candidates are
   inside 3 yd, pair to the nearest and continue. This is a variant, **not** the
   frozen rule's result: the frozen rule explicitly refuses tie-breaks.
3. **Margin distribution.** `second-nearest − nearest`, bucketed at
   `<0.25`, `0.25–0.5`, `0.5–1`, `>1` yd. This is the evidence for whether
   "nearest" is a discrimination or a coin flip.

**Precedence and N.** The headline stays `N = 2,295 = 1,731 frozen + 564 Route A`.
V4 is **not** a fifth slot: its turns are checked against Route A's, the overlap
is named, and the net new turns under a hypothetical precedence (frozen → V4 →
Route A) are printed. Nothing is added twice. Distances need frame-level
tracking; with none on this machine every table below prints `NOT EVALUABLE`
with its denominators and carries on.


In [ ]:
# The 870's geometry — the frozen 3-yd distance applied to every visible candidate
geom = aa.candidate_distance_table(multiple, TURNS, PLAYER_PLAY,
                                   tracking=(TRACKING or None))

print(f"ambiguous turns        : {len(geom):,}")
print(f"candidate rows         : {int(multiple.shape[0]):,}")
print(f"status                 : {geom['geometry_status'].iloc[0] if len(geom) else 'NO AMBIGUOUS TURNS'}")
if len(geom):
    print("per-turn status        : "
          + " · ".join(f"{k} {v:,}" for k, v in
                       geom["geometry_status"].value_counts().items()))
print()

within3 = aa.within_radius_distribution(geom)
margin = aa.margin_distribution(geom)

print(f"WITHIN {within3['radius_yd']:g} YD AT THE BREAK FRAME")
print(f"  evaluable {within3['n_evaluated']:,} of {within3['n_turns']:,} ambiguous "
      f"turns · {within3['n_candidate_rows_tracked']:,} tracked candidate rows")
print(f"  status: {within3['status']}")
w = pd.DataFrame(within3["buckets"])
print(w.assign(
    n_turns=w["n_turns"].map(lambda v: "n/a" if pd.isna(v) else f"{int(v):,}"),
    share_of_evaluated=w["share_of_evaluated"].map(
        lambda v: "n/a" if pd.isna(v) else f"{v:.1%}"),
    share_of_ambiguous=w["share_of_ambiguous"].map(
        lambda v: "n/a" if pd.isna(v) else f"{v:.1%}"),
).to_string(index=False))
print()

print("MARGIN — second-nearest minus nearest at the break frame")
print(f"  evaluable {margin['n_evaluated']:,} of {margin['n_turns']:,} ambiguous "
      f"turns · {margin['n_without_two_candidates']:,} with fewer than two tracked "
      f"candidates")
print(f"  status: {margin['status']}")
mg = pd.DataFrame(margin["buckets"])
print(mg.assign(
    n_turns=mg["n_turns"].map(lambda v: "n/a" if pd.isna(v) else f"{int(v):,}"),
    share_of_evaluated=mg["share_of_evaluated"].map(
        lambda v: "n/a" if pd.isna(v) else f"{v:.1%}"),
).to_string(index=False))
print(f"  median margin {margin['median_yd']:.3f} yd · mean {margin['mean_yd']:.3f} yd "
      f"(of {margin['n_evaluated']:,} evaluated turns)"
      if margin["n_evaluated"] else
      f"  median/mean margin: n/a of 0 evaluated turns")
print()
print("No bucket is a gate: every count above is printed with its denominator and "
      "nothing is selected by a margin.")

aa.note_run_status(
    STATUS_FILE, f"within_3yd/margin "
                 f"{within3['status'].split(' — ')[0]} / "
                 f"{within3['n_turns']:,} turns")


In [ ]:
# O3 amendment — V1-V3 beside V4, which is a TIE-BREAK and not the frozen rule
variant_rows = [aa.variant_fit(frame, rule, "primary") for rule in aa.PAIRING_RULES]
v4 = aa.nearest_variant(frame, geom, route_turns=route_turns)
variant_rows.append(v4)

variants = pd.DataFrame(variant_rows)
show = variants[["variant", "n", "beta", "lo", "hi", "label"]].copy()
show["beta"] = show["beta"].map(lambda v: "n/a" if pd.isna(v) else f"{v:+.3f}")
show["lo"] = show["lo"].map(lambda v: "n/a" if pd.isna(v) else f"{v:+.3f}")
show["hi"] = show["hi"].map(lambda v: "n/a" if pd.isna(v) else f"{v:+.3f}")
print(show.to_string(index=False))
print()
for row in variant_rows:
    print(f"  {row['variant']}: selects {row['population']}")
print()
print(f"V4 status: {v4['status']}")
print(f"  own n {v4['n']:,} rows · {v4['n_frozen_turns']:,} frozen + "
      f"{v4['n_nearest_turns']:,} nearest-resolved ambiguous turns")
print(f"  V4 label: {v4['label']}")
print(f"  {v4['attribution']}")

aa.note_run_status(
    STATUS_FILE, f"v4 {v4['status'].split(' — ')[0]} / "
                 f"{v4['n_ambiguous_turns']:,} ambiguous turns")


In [ ]:
# §0 — the one number, with its composition underneath. It is the four slots the
# pipeline computed in precedence order; nothing is added to it here.
by_rule = dict(assembled["route_a_by_rule"])
coverage = dict(assembled["coverage"])

print(f"N = {assembled['N']:,} of {band_turns:,} band turns "
      f"({assembled['share_of_band']:.1%})  ·  "
      f"tracking: {TRACKING_STATUS['status']}")
print(f"slots disjoint: {assembled['disjoint']} · turns in more than one slot "
      f"{assembled['n_double_counted']:,} of {assembled['N']:,}")
aa.note_run_status(STATUS_FILE, f"N = {assembled['N']:,}")

# ---- what this run wrote, beside the number -------------------------
# Every table this notebook computes is written, into the SAME directory
# the script writes: a table that existed only in memory is how a reader
# came to open a committed file this run never produced.
TABLES = {
    "funnel.csv": funnel,
    "multi_turns.csv": aa.multi_turn_table(multiple),
    "bounds.csv": bounds,
    # V1-V3 on every specification, V4 in its own table: the same 12 rows
    # the script writes, from the same function.
    "variants.csv": pd.DataFrame([aa.variant_fit(frame, rule, spec)
                                  for spec in aa.SPECS
                                  for rule in aa.PAIRING_RULES]),
    "composition.csv": aa.composition(single, multiple),
    "random_pairings.csv": pd.concat(
        [aa.random_assignments(frame, spec, reps=REPS, seed=SEED)
         for spec in aa.SPECS], ignore_index=True),
    "geometry.csv": geom,
    "route_a_turns.csv": route_turns,
    "route_a_audit.csv": route_audit,
    "assembly.csv": pd.DataFrame(assembled["composition"]),
    "single_turn_geometry.csv": singles["rows"],
    "within_3yd.csv": pd.DataFrame(within3["buckets"]),
    "margin_buckets.csv": pd.DataFrame(margin["buckets"]),
    "v4_nearest.csv": pd.DataFrame([v4]),
    "precedence.csv": precedence,
}
if WRITE_ARTEFACTS:
    written = aa.write_artefacts(OUT, TABLES)
    aa.write_manifest(
        OUT,
        [TURNS, PLAYER_PLAY, PLAYERS,
         *[f["path"] for f in TRACKING_STATUS["files"]]],
        written,
        config={"mode": "notebook", "reps": REPS, "seed": SEED,
                "specs": list(aa.SPECS),
                "pairing_rules": list(aa.PAIRING_RULES),
                "radii": list(RADII), "tracking": TRACKING},
        extra={"stage": "complete", "provenance": "notebook",
               "tracking": ("found" if TRACKING_STATUS["status"] == "FOUND"
                            else "none found"),
               "tracking_status": TRACKING_STATUS})
    written.append(OUT / "manifest.json")
    print(f"outputs: {OUT} — {len(written) + 1} files written by this run "
          f"({len(aa.ARTEFACT_TABLES)} tables + run_status.txt + "
          f"manifest.json)")
    print("  " + " · ".join([p.name for p in written]
                             + [STATUS_FILE.name]))
    aa.note_run_status(STATUS_FILE, f"wrote {len(written) + 1} files")
else:
    print(f"outputs: {OUT} — NOTHING WRITTEN (WRITE_ARTEFACTS is False); "
          f"the {len(aa.ARTEFACT_TABLES)} tables exist only in memory")
print()
print(pd.DataFrame(assembled["composition"])[
    ["slot", "label", "n", "share_of_N", "share_of_band"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"RULE share of N      {assembled['rule_share_of_N']:.1%}")
print(f"INFERENCE share of N {assembled['inference_share_of_N']:.1%}")
print(f"the frozen slot is the {len(single):,} single-candidate turns (RULE, no "
      f"disambiguation needed); the {len(route_turns):,} recovered turns are ones it "
      f"did not take, each labelled INFERENCE")
if by_rule:
    print("Route A by rule: " + " · ".join(f"{k} {v:,}" for k, v in by_rule.items()))
if coverage:
    print("coverage label: " + " · ".join(f"{k} {v:,}" for k, v in coverage.items()))

# §4's cross-check is run on the reference pass: the slot's turns are disjoint
# from the radii's by construction, and asking two disjoint sets whether they
# agree would be a question with no rows.
agree = aa.agreement(route_reference_turns, geo["rows"])
if len(agree) and int(agree.loc[agree["scope"] == "overall", "n_both"].sum()):
    print()
    print("Cross-check, turns where both routes fired:")
    print(agree.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
else:
    print()
    print("Cross-check where both routes fired: NOT EVALUABLE — "
          + ("no tracking on this machine, so no turn was paired by both routes."
             if TRACKING_STATUS["status"] != "FOUND" else
             "no turn was paired by both routes on this run."))

# V4 under the headline: not a slot, and its overlap with Route A is named
print()
print("V4 (INFERENCE — TIE-BREAK, not a slot of N)")
print(f"  turns with >=2 candidates inside 3 yd : {v4['n_nearest_turns']:,} of "
      f"{v4['n_ambiguous_turns']:,} ambiguous turns ({v4['status']})")
print(f"  overlap with Route A's {len(route_turns):,} turns        : "
      f"{v4['overlap_route_a']:,}")
print(f"  new turns if V4 took precedence        : "
      f"{v4['n_new_if_precedence']:,}  (precedence would be frozen -> V4 -> Route A)")
print(f"  headline N, its four slots             : {assembled['N']:,} = "
      f"{assembled['slots']['n_frozen']:,} frozen + "
      f"{assembled['slots']['n_geo_5']:,} geo(5) + "
      f"{assembled['slots']['n_geo_8']:,} geo(8) + "
      f"{assembled['slots']['n_route_a']:,} Route A")

aa.note_run_status(STATUS_FILE, "complete")


In [ ]:
# AC10 — the two-receiver anchor, recomputed from player_play.csv
align = aa.alignment_counts(PLAYER_PLAY)
print(align[["pair", "mixed", "n_plays", "all_plays", "spec_anchor", "matches"]]
      .to_string(index=False))
print()
print("'matches' compares this recomputation with the anchor the spec quotes; "
      "a False is printed, never adjusted away.")

### What this cannot say

1. **Which man he covered.** No field identifies the opponent; every slot except
   `N_frozen` is an `INFERENCE`.
2. **That `N` is a coverage count.** It is a count of attributions with stated
   evidence; the `RULE` and `INFERENCE` shares are printed beside it.
3. **Anything about the turns where the cohort hides candidates** — they are
   outside this cycle and outside every denominator here.
4. **That the geometric slots are free.** With no full tracking they are zero
   with `NOT EVALUABLE`; with tracking, the agreement rate printed above is the
   error rate to quote for the no-tracking turns.
5. `two_candidate_separation_gap_yd` is MAX(sep) − MIN(sep) of the candidates'
   separation values, not a distance between two players.